# 01 · Criação do Ambiente e Ingestão Bronze (V-Credit)

A ideia desse notebook é:
1. **Configurar o ambiente:** criar o catálogo, os schemas da arquitetura medalhão (`bronze`, `silver`, `gold`) e o volume onde ficam os CSVs;
2. **Ingerir as bases dimensionais:** carregar `base_atendentes.csv` e `base_motivos.csv` direto pra tabelas Delta Lake.


In [0]:
from pyspark.sql.functions import current_timestamp

catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Ambiente configurado com sucesso!")
print(f"Catálogo: {catalog}")
print(f"Caminho dos CSVs (Volume): {landing_path}")

Ambiente configurado com sucesso!
Catálogo: vcredit_analytics
Caminho dos CSVs (Volume): /Volumes/vcredit_analytics/bronze/inputs


---
## Ingestão da Base de Atendentes (`bronze.tb_atendentes`)

Puxando o arquivo `base_atendentes.csv` sem mexer no conteúdo, apenas exibindo a hora de ingestão.


In [0]:
# 1. Caminho e leitura do CSV
path_base_atendentes = f"{landing_path}/base_atendentes.csv"
df_base_atendentes_raw = spark.read.csv(path_base_atendentes, header=True, inferSchema=False)

# 2. Ingestão Delta Lake com timestamp de auditoria
(
    df_base_atendentes_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.bronze.tb_atendentes")
)

print(f"Tabela {catalog}.bronze.tb_atendentes salva!")
print(f"Total de atendentes carregados: {df_base_atendentes_raw.count()}")

display(spark.table(f"{catalog}.bronze.tb_atendentes").limit(5))

Tabela vcredit_analytics.bronze.tb_atendentes salva!
Total de atendentes carregados: 25


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente,ingestion_datetime
1,Marcela França,1,18/04/2019,16/11/2025,inativo,2026-10-26T17:02:40.117Z
2,Zilda Esteves,1,12/09/2023,24/01/2026,inativo,2026-10-26T17:02:40.117Z
3,William Marques,1,28/05/2019,15/04/2025,inativo,2026-10-26T17:02:40.117Z
4,Solange Silveira,Nível 1,07/10/2019,null,ativo,2026-10-26T17:02:40.117Z
5,Joaquim Barros,Nível 1,06/12/2020,23/05/2026,inativo,2026-10-26T17:02:40.117Z


---
## Ingestão da Base de Motivos (`bronze.tb_motivos`)

Agora puxamos a `base_motivos.csv` seguindo a mesma lógica da Bronze.


In [0]:
# 1. Caminho e leitura do CSV
path_base_motivos = f"{landing_path}/base_motivos.csv"
df_base_motivos_raw = spark.read.csv(path_base_motivos, header=True, inferSchema=False)

# 2. Ingestão Delta Lake com timestamp de auditoria
(
    df_base_motivos_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.bronze.tb_motivos")
)

print(f"Tabela {catalog}.bronze.tb_motivos salva!")
print(f"Total de motivos carregados: {df_base_motivos_raw.count()}")

display(spark.table(f"{catalog}.bronze.tb_motivos").limit(5))

Tabela vcredit_analytics.bronze.tb_motivos salva!
Total de motivos carregados: 14


id_motivo,nome_motivo,categoria,criticidade,ingestion_datetime
11,Bloqueio de cartão,Cartao,Alta,2026-10-26T17:03:12.837Z
7,Compra não autorizada,Cartao,alta,2026-10-26T17:03:12.837Z
13,Dúvidas gerais sobre programa de pontos,Beneficios,baixa,2026-10-26T17:03:12.837Z
14,Transferência de agência,null,Media,2026-10-26T17:03:12.837Z
9,Problema com aplicativo,Atendimento,Media,2026-10-26T17:03:12.837Z


Assim, as duas bases brutas estão salvas em Delta Lake com metadados de auditoria.
Ao rodar este notebook, ele não vai gerar registros duplicados (totalmente idempotente).

